In [ ]:
pip install tifffile scikit-image scikit-learn tensorflow numpy

In [ ]:
dir = "C:/Users/FSOS/Downloads/ai-first-nations-main/ai-first-nations-main/PhDMangroveDataset"

In [ ]:
#reading the tiff files and putting it into labelled, x and y

X = []
y = []
target_shape = (256, 256, 7) #this is the file size of the satilite data (there are 7 bands of information attached, TLDR WHEN I LEARN THIS DATA MORE WE CAN USE IT FOR OTHER MODELS)
class_map = {"Mangroves": 1, "NonMangroves": 0}

for class_name, label in class_map.items():
    class_dir = os.path.join(dir, class_name)
    for fname in os.listdir(class_dir):
        if fname.endswith('.tif') or fname.endswith('.tiff'):
            image = tifffile.imread(os.path.join(class_dir, fname))
            if image.shape != target_shape:
                image = resize(image, target_shape, anti_aliasing=True)
            X.append(image)
            y.append(label)

X = np.array(X)
y = np.array(y)
print(f"X shape: {X.shape}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

In [ ]:
#more efficent method to normalise
normalisation_l = tf.keras.layers.Normalization(axis=-1)
normalisation_l.adapt(X_train)

In [ ]:
def create_cnn(input_shape=(256, 256, 9)):
    model = Sequential([
        normalisation_l,
        layers.RandomFlip("horizontal_and_vertical", input_shape=input_shape),
        layers.RandomRotation(0.2),


        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),


        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),


        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),


        layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),


        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

In [ ]:
cnn_supervised = create_cnn()
cnn_supervised.fit(X_train, y_train, epochs=75, batch_size=256, validation_split=0.2)